# 实验三参考答案：Python 工程化改造实战

> 对应讲次：第3讲 Spec-driven 协作建模 × Python 工程化
>
> 本参考答案展示完整的工程化改造流程和 Spec-driven 开发实战

## 任务1：模块拆分（20分）

In [ ]:
import os
from pathlib import Path

# 创建项目结构
base = Path(".")
dirs = ["sci_analyzer", "tests", "output"]
for d in dirs:
    (base / d).mkdir(exist_ok=True)

# === sci_analyzer/__init__.py ===
(base / "sci_analyzer" / "__init__.py").write_text(
    '"""科研实验数据管理与分析工具"""\n\n'
    'from .loader import load_data\n'
    'from .cleaner import clean_data, detect_outliers\n'
    'from .stats_engine import run_statistics\n'
    'from .exceptions import DataError, LoadError, CleanError\n\n'
    '__version__ = "0.1.0"\n'
    '__all__ = [\n'
    '    "load_data", "clean_data", "detect_outliers",\n'
    '    "run_statistics", "DataError", "LoadError", "CleanError",\n'
    ']\n'
)

# === sci_analyzer/exceptions.py ===
(base / "sci_analyzer" / "exceptions.py").write_text(
    '"""自定义异常层级"""\n\n\n'
    'class DataError(Exception):\n'
    '    """数据处理相关错误基类"""\n'
    '    pass\n\n\n'
    'class LoadError(DataError):\n'
    '    """数据加载失败"""\n'
    '    pass\n\n\n'
    'class CleanError(DataError):\n'
    '    """数据清洗失败"""\n'
    '    def __init__(self, column: str, reason: str):\n'
    '        self.column = column\n'
    '        super().__init__(f"列\'{column}\'清洗失败: {reason}")\n'
)

print("项目结构已创建:")
for root, dirs_list, files in os.walk("sci_analyzer"):
    level = root.replace("sci_analyzer", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

### loader.py 实现

In [ ]:
loader_code = """from pathlib import Path
from typing import Optional
import pandas as pd
import logging

from .exceptions import LoadError

logger = logging.getLogger(__name__)

SUPPORTED_FORMATS = {".csv", ".xlsx", ".xls", ".json"}


def load_data(
    file_path: Path,
    sheet_name: Optional[str] = None,
    encoding: str = "utf-8-sig",
) -> pd.DataFrame:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在: {file_path}")

    suffix = file_path.suffix.lower()
    if suffix not in SUPPORTED_FORMATS:
        raise ValueError(f"不支持的格式: {suffix}, 仅支持 {SUPPORTED_FORMATS}")

    try:
        if suffix == ".csv":
            try:
                return pd.read_csv(file_path, encoding=encoding)
            except UnicodeDecodeError:
                logger.warning(f"{encoding} 解码失败，降级为 gbk")
                return pd.read_csv(file_path, encoding="gbk")
        elif suffix in (".xlsx", ".xls"):
            return pd.read_excel(file_path, sheet_name=sheet_name)
        elif suffix == ".json":
            return pd.read_json(file_path)
    except (FileNotFoundError, ValueError):
        raise
    except Exception as e:
        raise LoadError(f"加载 {file_path.name} 失败: {e}") from e
"""

(Path("sci_analyzer") / "loader.py").write_text(loader_code)
print("loader.py 已写入")
print("特点: 类型注解完整 / 自动编码降级 / 自定义 LoadError")

### cleaner.py 实现

In [ ]:
cleaner_code = """from typing import Any, Optional
import pandas as pd
import numpy as np
from scipy import stats as sp_stats
import logging

from .exceptions import CleanError

logger = logging.getLogger(__name__)


def clean_data(
    df: pd.DataFrame,
    drop_duplicates: bool = True,
    fill_strategy: str = "median",
    outlier_method: Optional[str] = None,
    outlier_threshold: float = 1.5,
) -> pd.DataFrame:
    result = df.copy()

    if drop_duplicates:
        before = len(result)
        result = result.drop_duplicates()
        removed = before - len(result)
        if removed > 0:
            logger.info(f"去除 {removed} 行重复数据")

    numeric_cols = result.select_dtypes(include="number").columns
    if fill_strategy == "median":
        result[numeric_cols] = result[numeric_cols].fillna(result[numeric_cols].median())
    elif fill_strategy == "mean":
        result[numeric_cols] = result[numeric_cols].fillna(result[numeric_cols].mean())
    elif fill_strategy == "zero":
        result[numeric_cols] = result[numeric_cols].fillna(0)

    if outlier_method:
        for col in numeric_cols:
            mask, _ = detect_outliers(result[col], method=outlier_method, threshold=outlier_threshold)
            result.loc[mask, col] = np.nan
        result[numeric_cols] = result[numeric_cols].fillna(result[numeric_cols].median())

    return result


def detect_outliers(
    series: pd.Series,
    method: str = "iqr",
    threshold: float = 1.5,
) -> tuple[pd.Series, dict[str, Any]]:
    values = series.dropna()
    if method == "iqr":
        Q1, Q3 = values.quantile(0.25), values.quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - threshold * IQR, Q3 + threshold * IQR
        mask = (series < lower) | (series > upper)
        info = {"method": "iqr", "Q1": Q1, "Q3": Q3, "IQR": IQR, "lower": lower, "upper": upper}
    elif method == "zscore":
        z = np.abs(sp_stats.zscore(values))
        z_full = pd.Series(np.nan, index=series.index)
        z_full[values.index] = z
        mask = z_full > threshold
        info = {"method": "zscore", "threshold": threshold}
    else:
        raise CleanError("series", f"不支持的方法: {method}")

    info["outlier_count"] = int(mask.sum())
    return mask, info
"""

(Path("sci_analyzer") / "cleaner.py").write_text(cleaner_code)
print("cleaner.py 已写入")
print("特点: 不修改原输入(.copy()) / 策略可配置 / 返回 tuple")

### stats_engine.py 实现

In [ ]:
stats_code = """from typing import Any
import pandas as pd
from scipy import stats as sp_stats
import logging

logger = logging.getLogger(__name__)


def run_statistics(
    df: pd.DataFrame,
    target_col: str = None,
    group_col: str = None,
) -> dict[str, Any]:
    result = {}
    numeric_cols = df.select_dtypes(include="number").columns.tolist()

    # 描述统计
    result["describe"] = df[numeric_cols].describe().to_dict()

    # 相关性矩阵
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr()
        result["correlation"] = corr.to_dict()

        # 高相关对
        high_corr = []
        for i in range(len(corr)):
            for j in range(i + 1, len(corr)):
                r = corr.iloc[i, j]
                if abs(r) > 0.7:
                    high_corr.append((corr.index[i], corr.columns[j], round(r, 4)))
        result["high_correlation_pairs"] = high_corr

    # 分组对比 t 检验
    if target_col and group_col and group_col in df.columns:
        groups = df[group_col].unique()
        if len(groups) == 2:
            g1 = df[df[group_col] == groups[0]][target_col].dropna()
            g2 = df[df[group_col] == groups[1]][target_col].dropna()
            t_stat, p_val = sp_stats.ttest_ind(g1, g2)
            result["t_test"] = {
                "groups": list(groups),
                "t_statistic": round(t_stat, 4),
                "p_value": round(p_val, 6),
                "significant": p_val < 0.05,
            }

    return result
"""

(Path("sci_analyzer") / "stats_engine.py").write_text(stats_code)
print("stats_engine.py 已写入")

In [ ]:
# 创建空的 visualizer.py (简化版)
viz_code = """from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import logging

logger = logging.getLogger(__name__)


def plot_results(
    df: pd.DataFrame,
    output_dir: Path = Path("output"),
) -> list[Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    saved = []
    numeric_cols = df.select_dtypes(include="number").columns[:3]

    for col in numeric_cols:
        fig, ax = plt.subplots(figsize=(8, 4))
        df[col].hist(ax=ax, bins=20)
        ax.set_title(col)
        path = output_dir / f"{col}_hist.png"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        saved.append(path)

    return saved
"""

(Path("sci_analyzer") / "visualizer.py").write_text(viz_code)
print("visualizer.py 已写入")
print()

# 验证导入
import importlib, sys
sys.path.insert(0, ".")
try:
    mod = importlib.import_module("sci_analyzer")
    print(f"导入成功: sci_analyzer v{mod.__version__}")
    print(f"公开接口: {mod.__all__}")
except Exception as e:
    print(f"导入失败: {e}")

## 任务2：类型注解 + 异常处理（25分）

上面的 loader.py 和 cleaner.py 已包含完整的类型注解和异常处理。

**类型注解要点**：
- `file_path: Path` — 使用 pathlib 而非字符串
- `Optional[str]` — 标记可选参数
- `-> tuple[pd.Series, dict[str, Any]]` — 精确的返回类型

**异常处理要点**：
- 自定义三层异常：`DataError > LoadError / CleanError`
- 编码错误自动降级（utf-8 → gbk）
- 不使用 bare except，每个 except 都有具体类型

## 任务3：测试用例（25分）

In [ ]:
import textwrap

# conftest.py - 共享 fixture
conftest = """import pytest
import pandas as pd
import numpy as np
from pathlib import Path


@pytest.fixture
def sample_df():
    return pd.DataFrame({
        "name": ["A", "B", "C", "A", "D"],
        "value": [1.0, 2.0, np.nan, 1.0, 100.0],
        "category": ["x", "y", "x", "x", "y"],
    })


@pytest.fixture
def csv_file(tmp_path):
    f = tmp_path / "test.csv"
    f.write_text("name,value\nA,1\nB,2\nC,3", encoding="utf-8-sig")
    return f


@pytest.fixture
def excel_file(tmp_path):
    f = tmp_path / "test.xlsx"
    pd.DataFrame({"a": [1, 2], "b": [3, 4]}).to_excel(f, index=False)
    return f
"""

(Path("tests") / "__init__.py").touch()
(Path("tests") / "conftest.py").write_text(conftest)
print("tests/conftest.py 已写入（3个fixture）")

In [ ]:
# test_loader.py
test_loader = """import pytest
from pathlib import Path
from sci_analyzer.loader import load_data
from sci_analyzer.exceptions import LoadError


def test_load_csv(csv_file):
    df = load_data(csv_file)
    assert len(df) == 3
    assert "name" in df.columns


def test_load_excel(excel_file):
    df = load_data(excel_file)
    assert len(df) == 2
    assert list(df.columns) == ["a", "b"]


def test_load_nonexistent():
    with pytest.raises(FileNotFoundError):
        load_data(Path("ghost.csv"))


def test_load_unsupported_format(tmp_path):
    f = tmp_path / "test.txt"
    f.write_text("hello")
    with pytest.raises(ValueError, match="不支持的格式"):
        load_data(f)
"""

(Path("tests") / "test_loader.py").write_text(test_loader)
print("tests/test_loader.py 已写入（4个测试）")

In [ ]:
# test_cleaner.py
test_cleaner = """import pytest
import pandas as pd
import numpy as np
from sci_analyzer.cleaner import clean_data, detect_outliers
from sci_analyzer.exceptions import CleanError


def test_clean_removes_duplicates(sample_df):
    result = clean_data(sample_df, drop_duplicates=True)
    assert len(result) == 4  # "A" 重复，去重后少1行


def test_clean_fills_missing(sample_df):
    result = clean_data(sample_df, fill_strategy="median")
    assert result["value"].isnull().sum() == 0


def test_clean_does_not_modify_input(sample_df):
    original_len = len(sample_df)
    _ = clean_data(sample_df)
    assert len(sample_df) == original_len  # 原始未被修改


@pytest.mark.parametrize("method,threshold,expected_min", [
    ("iqr", 1.5, 0),
    ("zscore", 2.0, 0),
])
def test_detect_outliers_methods(sample_df, method, threshold, expected_min):
    mask, info = detect_outliers(sample_df["value"].dropna(), method=method, threshold=threshold)
    assert mask.sum() >= expected_min
    assert "outlier_count" in info


def test_detect_outliers_invalid_method(sample_df):
    with pytest.raises(CleanError):
        detect_outliers(sample_df["value"], method="invalid")
"""

(Path("tests") / "test_cleaner.py").write_text(test_cleaner)
print("tests/test_cleaner.py 已写入（5个测试，含参数化）")

In [ ]:
# 运行 pytest
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
print(f"\n退出码: {result.returncode} ({'全部通过' if result.returncode == 0 else '有失败'})")

## 任务4：Spec-driven 新功能 — reporter 模块（30分）

### speckit.md 内容

```markdown
# speckit.md - Reporter 模块规约

## Constitution（硬约束）

1. 所有公开函数必须有类型注解（参数+返回值）
2. 所有公开函数必须有 Google style docstring
3. 不修改传入的 DataFrame（只读操作或 .copy()）
4. 不允许 bare except
5. 文件操作用 pathlib.Path
6. 数值保留 4 位小数
7. 中文注释，英文变量名 snake_case

## Context

- Python 3.12, pandas 2.x, pathlib
- 依赖模块：stats_engine.py 的输出作为输入
- 输出：Markdown 格式的分析报告

## Task: reporter.py

### 接口
def generate_report(df, stats_result, output_path) -> Path

### 行为
1. 生成 .md 报告文件
2. 报告含 4 段：数据概览 / 清洗摘要 / 统计结果 / 结论

### 验收标准
- 输出文件存在且 >= 100 字符
- Markdown 语法正确
- pytest 测试通过
```

In [ ]:
# 按 Spec 实现 reporter.py
reporter_code = """from pathlib import Path
from typing import Any
import pandas as pd
import logging

logger = logging.getLogger(__name__)


def generate_report(
    df: pd.DataFrame,
    stats_result: dict[str, Any],
    output_path: Path,
) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    sections = []

    # 1. 数据概览
    sections.append("# 数据分析报告\n")
    sections.append("## 1. 数据概览\n")
    sections.append(f"- 数据形状: {df.shape[0]} 行 x {df.shape[1]} 列")
    sections.append(f"- 数值列: {len(df.select_dtypes(include='number').columns)}")
    sections.append(f"- 缺失值总数: {df.isnull().sum().sum()}")
    sections.append("")

    # 2. 清洗摘要
    sections.append("## 2. 数据质量\n")
    sections.append(f"- 重复行: {df.duplicated().sum()}")
    missing_cols = df.columns[df.isnull().any()].tolist()
    sections.append(f"- 含缺失值的列: {len(missing_cols)}")
    sections.append("")

    # 3. 统计结果
    sections.append("## 3. 统计分析结果\n")
    if "high_correlation_pairs" in stats_result:
        pairs = stats_result["high_correlation_pairs"]
        sections.append(f"- 高相关变量对(|r|>0.7): {len(pairs)} 对")
    if "t_test" in stats_result:
        t = stats_result["t_test"]
        sig = "显著" if t["significant"] else "不显著"
        sections.append(f"- t检验: t={t['t_statistic']}, p={t['p_value']} ({sig})")
    sections.append("")

    # 4. 结论
    sections.append("## 4. 结论\n")
    sections.append(f"本报告基于 {df.shape[0]} 条数据记录生成。")
    sections.append("数据质量良好，统计分析结果可靠。")

    report_text = "\n".join(sections)
    output_path.write_text(report_text, encoding="utf-8")
    logger.info(f"报告已生成: {output_path} ({len(report_text)} 字符)")
    return output_path
"""

(Path("sci_analyzer") / "reporter.py").write_text(reporter_code)
print("sci_analyzer/reporter.py 已按 Spec 实现")
print("符合 Constitution: 类型注解/docstring/不修改输入/pathlib/具体异常")

In [ ]:
# 测试 reporter
test_reporter = """import pytest
import pandas as pd
from pathlib import Path
from sci_analyzer.reporter import generate_report


@pytest.fixture
def report_input():
    df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
    stats = {"high_correlation_pairs": [("a", "b", 0.99)], "t_test": {"t_statistic": 2.5, "p_value": 0.02, "significant": True}}
    return df, stats


def test_report_generates_file(report_input, tmp_path):
    df, stats = report_input
    out = tmp_path / "report.md"
    result = generate_report(df, stats, out)
    assert result.exists()
    assert result.stat().st_size > 100


def test_report_contains_sections(report_input, tmp_path):
    df, stats = report_input
    out = tmp_path / "report.md"
    generate_report(df, stats, out)
    content = out.read_text()
    assert "数据概览" in content
    assert "统计分析" in content
    assert "结论" in content


def test_report_empty_stats(tmp_path):
    df = pd.DataFrame({"x": [1, 2]})
    out = tmp_path / "report.md"
    result = generate_report(df, {}, out)
    assert result.exists()
"""

(Path("tests") / "test_reporter.py").write_text(test_reporter)
print("tests/test_reporter.py 已写入（3个测试）")

In [ ]:
# 最终验证：全部测试
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
print(f"\n退出码: {result.returncode} ({'全部通过!' if result.returncode == 0 else '有失败'})")

## 思考题参考答案

### 1. Spec-driven vs 两段式 Prompt 的优势与代价

**优势**：
- 一致性：多轮对话中 AI 始终遵守同一套规则
- 可验收：Constitution 的每条规则都可以通过 code review 或测试检查
- 可复用：同一个 speckit.md 可用于多个模块的实现

**代价**：
- 前期投入大：写好 Constitution 和 Tasks 需要时间
- 灵活性低：探索阶段不适合用 Spec（过早约束会限制创意）
- 维护成本：项目演进时 Spec 也需要同步更新

**选择原则**：快速原型用 Prompt；正式工程用 Spec。

### 2. 最有效的 Constitution 规则

"不修改原始输入数据（必须 .copy()）"最有效。原因：
- AI 默认倾向于就地修改（性能更好）
- 但这会导致难以调试的副作用 bug
- 有了这条规则后，AI 每次都会加 `.copy()`，彻底消除隐患

### 3. AI 实现未通过测试的修正策略

1. 定位失败：先看 pytest 输出哪个测试失败、报什么错
2. 精准反馈：把错误信息完整复制给 AI，附带"请修改 X 函数的 Y 处"
3. 不泛化：只修问题所在，不让 AI 重写整个文件
4. 验证循环：修完立即重跑 pytest，不通过就继续 Round 3